# Notebook to test quality of binding_db's data
Download from: https://www.bindingdb.org/rwd/bind/chemsearch/marvin/Download.jsp

In [45]:
%pip install pandas numpy rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 80.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 96.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, re, warnings
import pandas as pd
import numpy as np

In [8]:
datasets = os.path.join(os.getcwd(), "datasets")
print(f"Current Working directory: {os.getcwd()}")
print(f"Datasets available in directory: {datasets}:\n\t{os.listdir(datasets)}")

Current Working directory: /Users/michaeljopiti/Prototype/binding_db
Datasets available in directory: /Users/michaeljopiti/Prototype/binding_db/datasets:
	['ChEBI_Results.tsv', 'BindingDB_ChEMBL_202503_tsv.zip', 'BindingDB_PDSPKi_202503_tsv.zip', 'BindingDB_ChEMBL.tsv', 'BindingDB_PDSPKi.tsv']


In [ ]:
# File read is from request to ChEBI database with steroid + is a chemical entity
file = os.path.join(datasets, "ChEBI_Results.tsv")
print(f"Reading file: {file}")
# df = pd.read_csv(file, sep='\t', on_bad_lines='warn') # on_bad_lines='warn' , nrows=1_000_000
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    # Warning: it seems that in the file there are lines with more columns than expected
        # Probably due to the fact that some characters are not properly escaped when callind pf.read_csv

    # For now it is not significant, as there are <100 lines escaped, meaning 0.01% of the total content
    # Momentarily ignoring the warnings lines and keeping the data as is as we only look for 'Ligand SMILES'

    with open(file) as f:
        header = f.readline().strip().split('\t')
        print(f"\t📋 Header fields: {header}\n\t\t\tlength: {len(header)}")

    # Load file without header, then assign it manually, some stupid ass error of formatting occurs and it shift all the cols names to the right
    df = pd.read_csv(file, sep='\t', header=None, skiprows=1, names=header, engine='python', index_col=False)

    print(f"\t📊 Dataframe shape: {df.shape}")
    print(f"\t⚠️ Total warnings: {len(w)}")
    print(f"\t🚨 Percentage skipped: {len(w)/len(df)*100:.2f}%")

df.head()


Reading file: /Users/michaeljopiti/Prototype/binding_db/datasets/ChEBI_Results.tsv
	📋 Header fields: ['ID', 'NAME', 'DEFINITION', 'STAR', 'SECONDARY ID', 'SYNONYM', 'IUPAC NAME', 'INN', 'FORMULA', 'MASS', 'MONOISOTOPIC MASS', 'CHARGE', 'SMILES', 'INCHI', 'INCHIKEY', 'BPDB ACCESSION', 'CHEMIDPLUS', 'KEGG COMPOUND ACCESSION', 'KEGG DRUG ACCESSION', 'KEGG GLYCAN ACCESSION', 'PUBMED CITATION']
			length: 21
	📊 Dataframe shape: (666, 21)
	⚠️ Total warnings: 0
	🚨 Percentage skipped: 0.00%


,ID,NAME,DEFINITION,STAR,SECONDARY ID,SYNONYM,IUPAC NAME,INN,FORMULA,MASS,...,CHARGE,SMILES,INCHI,INCHIKEY,BPDB ACCESSION,CHEMIDPLUS,KEGG COMPOUND ACCESSION,KEGG DRUG ACCESSION,KEGG GLYCAN ACCESSION,PUBMED CITATION
0,CHEBI:177917,phenolic steroid,NaN,3,CHEBI:8074; CHEBI:25968,phenolic steroids; a phenolic steroid,NaN,NaN,C18H23OR,255.375,...,0.0,CC12CCC3C(CCC4=CC(O)=CC=C34)C1CCC2[*],NaN,NaN,NaN,NaN,C02453,NaN,NaN,13752799; 4303459; 4622133; 4753431; 4798212; ...
1,CHEBI:177918,phenolic steroid 3-O-sulfate(1-),NaN,2,NaN,a phenolic steroid 3-O-sulfate,NaN,NaN,C18H22O4SR,334.432,...,-1.0,CC12CCC3C(CCC4=CC(OS([O-])(=O)=O)=CC=C34)C1CCC...,NaN,NaN,NaN,NaN,C02590,NaN,NaN,NaN
2,CHEBI:138029,14alpha-methyl steroid,Any steroid carrying a 14α-methyl substituent.,3,NaN,a 14alpha-methyl steroid,NaN,NaN,C20H33R,273.477,...,0.0,C12C([C@]3(C(C(CC3)*)(C)CC1)C)CCC4C2(CCCC4)C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CHEBI:176901,14alpha-hydroxymethyl steroid,Any steroid carrying a 14α-hydroxymethyl subst...,2,NaN,a 14alpha-hydroxymethyl steroid,NaN,NaN,C20H33OR,289.476,...,0.0,C12C([C@]3(C(C(CC3)*)(C)CC1)CO)CCC4C2(CCCC4)C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CHEBI:176903,14alpha-dihydroxymethyl steroid,Any steroid carrying a 14α-dihydroxymethyl sub...,2,NaN,a 14alpha-dihydroxymethyl steroid,NaN,NaN,C20H33O2R,305.476,...,0.0,C12C([C@]3(C(C(CC3)*)(C)CC1)C(O)O)CCC4C2(CCCC4)C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
for i, col in enumerate(df.columns):
    print(f"\t{i}: {col}")

In [57]:
missing = df['SMILES'].isna().sum()
total = len(df)
print(f"Total molecules: {total}")
print(f"{missing} molecules have no SMILES out of {total} total ({(missing/total)*100:.2f}%)")
# Trim does not work as expected, as the SMILES are not properly formatted
# Drop those which do not have a SMILES
df = df.dropna(subset=['SMILES'])
print(f"Removed molecules wiht NaN in SMILES, parsed df contains {len(df)/total*100:.2f}% of original dataframe")

# df.head()


Total molecules: 666
0 molecules have no SMILES out of 666 total (0.00%)
Removed molecules wiht NaN in SMILES, parsed df contains 100.00% of original dataframe


In [61]:
from rdkit import Chem
from rdkit.Chem import PandasTools

PandasTools.AddMoleculeColumnToFrame(df, smilesCol='SMILES')

steroid_core = Chem.MolFromSmarts("C1CCC2C(C1)CCC3C2CCCC3")  # A-B-C rings

print(f"Valid RDKit mols: {df['ROMol'].notna().sum()/len(df)*100:.2f}%")
df['has_steroid'] = df['ROMol'].apply(lambda mol: mol.HasSubstructMatch(steroid_core) if mol else False)
matches = df[df['has_steroid']]

print(f"✅ Found {len(matches)} steroid-like molecules, equivalent to {len(matches)/len(df)*100:.2f}% of total")


Valid RDKit mols: 100.00%
✅ Found 210 steroid-like molecules, equivalent to 31.53% of total


I wasn't satisfied with the 210/666 steroids I found, so I tried another approach with some type of scaffolding. No clue of how it actually works.

In [62]:
from rdkit.Chem.Scaffolds import MurckoScaffold

df['scaffold'] = df['ROMol'].apply(lambda m: MurckoScaffold.GetScaffoldForMol(m))
df['has_steroid_core'] = df['scaffold'].apply(lambda m: m.HasSubstructMatch(steroid_core))

matches = df[df['has_steroid_core']]
print(f"🎯 Matches via scaffold: {len(matches)}, aka {len(matches)/len(df)*100:.2f}% of total")


🎯 Matches via scaffold: 210, aka 31.53% of total
